## 1. Setup Mount & Drive

In [ ]:
import os
import re
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Path Configuration
BASE_DIR = "/content/drive/MyDrive/skripsi/dataset/mbg"
RAW_DATA_MBG_DIR = os.path.join(BASE_DIR, "raw_data", "mbg")
RAW_DATA_MAKAN_DIR = os.path.join(BASE_DIR, "raw_data", "makan_bergizi_gratis")
MERGED_OUTPUT_DIR = os.path.join(BASE_DIR, "merged_data")

os.makedirs(MERGED_OUTPUT_DIR, exist_ok=True)

print(f"📂 Folder Input MBG   : {RAW_DATA_MBG_DIR}")
print(f"📂 Folder Input Makan : {RAW_DATA_MAKAN_DIR}")
print(f"📂 Folder Output      : {MERGED_OUTPUT_DIR}")


## 2. Definisi Konstanta & Aturan Filter

In [ ]:
# --- TIMEZONE CONFIGURATION ---
START_DATE = pd.Timestamp("2025-01-06").tz_localize('UTC')
END_DATE   = pd.Timestamp("2026-01-07").tz_localize('UTC')

# --- CONFIGURATION KEYWORDS ---
KEYWORDS_MAKAN = [
    "makan bergizi gratis",
    "makan bergizi",
    "makan siang gratis",
    "program makan gratis"
]
KEYWORDS_MBG = ["mbg"]

# --- SPAM KEYWORDS ---
SPAM_KEYWORDS = [
    # Judi Online
    "slot", "gacor", "zeus", "pragmatic", "maxwin", "wd", "depo",
    "pola gacor", "rtp", "judol", "link daftar", "situs",
    # E-Commerce & Spam Bot
    "shopee", "tokopedia", "lazada", "blibli", "cek keranjang",
    "spaylater", "dana kaget", "giveaway", "racun", "link di bio",
    "affiliate", "open joki", "joki tugas", "jasjok", "convert pulsa",
    "mutualan", "follback", "biro jodoh",
    # Konten Dewasa
    "open bo", "vcs", "video syur", "museum", "boke", "desah", "colmek"
]

# Siapkan regex pattern untuk kecepatan filtering (Vektorisasi Pangdas)
SPAM_REGEX_PATTERN = '|'.join(SPAM_KEYWORDS)


## 3. Fungsi Modular Pipeline Pengolahan

In [ ]:
def load_and_merge_csv(folder_path):
    """
    Reads all CSV files in a given folder and merges them into a single DataFrame.
    """
    all_files = glob.glob(os.path.join(folder_path, "*.csv"))
    if not all_files:
        print(f"⚠️ No files found in {folder_path}")
        return pd.DataFrame()

    df_list = []
    for filepath in all_files:
        try:
            df = pd.read_csv(
                filepath,
                on_bad_lines='skip',
                lineterminator='\n',
                encoding_errors='ignore',
                low_memory=False
            )
            df_list.append(df)
        except Exception as e:
            print(f" ❌ Failed to read {os.path.basename(filepath)}: {e}")

    if not df_list:
        return pd.DataFrame()

    return pd.concat(df_list, ignore_index=True)


def standardize_columns(df):
    """
    Standardizes column names and normalizes corrupted anomalies (like username).
    """
    # Rename default mapping
    col_mapping = {'date': 'created_at', 'text': 'full_text', 'id_str': 'id'}
    df.rename(columns=col_mapping, inplace=True, errors='ignore')

    # Normalize username columns caused by crawling parse error
    anomaly_cols = ['username', 'username\r', 'username;;;;;;;\r']

    # Ensure primary username exists
    if 'username' not in df.columns:
        df['username'] = pd.NA

    # Coalesce via combine_first using vectors
    for col in anomaly_cols:
        if col in df.columns and col != 'username':
            df[col] = df[col].replace(r'^\s*$', pd.NA, regex=True)
            df['username'] = df['username'].replace(r'^\s*$', pd.NA, regex=True)
            df['username'] = df['username'].combine_first(df[col])

    # Drop dropped anomaly columns to clean the dataset
    cols_to_drop = [c for c in anomaly_cols if c != 'username']
    df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    return df


def filter_by_required_keywords(df, required_keywords):
    """
    Filters rows ensuring the full_text contains at least one of the required keywords.
    """
    pattern = '|'.join(required_keywords)
    mask = df['full_text'].astype(str).str.contains(pattern, case=False, na=False)
    return df[mask].copy()


def filter_by_date(df, start_date, end_date):
    """
    Transforms the date column to timezone-aware UTC and filters by date range.
    """
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce', format='mixed', utc=True)
    df.dropna(subset=['created_at'], inplace=True)
    mask = (df['created_at'] >= start_date) & (df['created_at'] < end_date)
    return df[mask].copy()


def remove_advanced_duplicates(df):
    """
    Aggressively removes duplicates by ignoring URLs, mentions, hashtags, and punctuations.
    """
    temp_series = df['full_text'].astype(str).str.lower()
    temp_series = temp_series.str.replace(r'http\S+|www\S+|https\S+', '', regex=True)
    temp_series = temp_series.str.replace(r'@\w+', '', regex=True)
    temp_series = temp_series.str.replace(r'#\w+', '', regex=True)
    temp_series = temp_series.str.replace(r'[^\w\s]', '', regex=True)
    temp_series = temp_series.str.replace(r'\s+', ' ', regex=True).str.strip()

    df['temp_dedup'] = temp_series

    df = df[df['temp_dedup'] != '']
    df.drop_duplicates(subset=['temp_dedup'], keep='first', inplace=True)
    df.drop(columns=['temp_dedup'], inplace=True)

    return df


def remove_spam(df):
    """
    Filters out spam rows relying on predefined spam keywords.
    """
    is_spam_mask = df['full_text'].astype(str).str.contains(SPAM_REGEX_PATTERN, case=False, na=False)
    return df[~is_spam_mask].copy()


def process_pipeline(folder_path, source_label, required_keywords):
    """
    The main coordinator function executing the established data processing pipeline.
    """
    print(f"\\n🚀 Processing Folder: {source_label} ...")

    # 1. Load Data
    df = load_and_merge_csv(folder_path)
    if df.empty:
        return df

    initial_count = len(df)
    print(f"   📥 Total Raw Data    : {initial_count} rows")

    # 2. Standardize & Fix Anomalies
    df = standardize_columns(df)
    if 'full_text' not in df.columns or 'created_at' not in df.columns:
        print("   ❌ Error: 'full_text' or 'created_at' column missing.")
        return pd.DataFrame()

    # 3. Filter Keywords
    df = filter_by_required_keywords(df, required_keywords)
    print(f"   🔍 Keyword Mismatch : {initial_count - len(df)} rows dropped")
    keyword_filtered_count = len(df)

    # 4. Filter Date Range
    df = filter_by_date(df, START_DATE, END_DATE)
    print(f"   📅 Out of Date Range : {keyword_filtered_count - len(df)} rows dropped")
    date_filtered_count = len(df)

    # 5. Remove Duplicates
    df = remove_advanced_duplicates(df)
    print(f"   ♻️  Duplicates Removed: {date_filtered_count - len(df)} rows (Advanced)")
    dedup_count = len(df)

    # 6. Remove Spam
    df = remove_spam(df)
    print(f"   🗑️  Spam Removed      : {dedup_count - len(df)} rows")

    # 7. Finalization
    df.sort_values(by='created_at', ascending=True, inplace=True)
    df['source_keyword'] = source_label

    print(f"   ✨ FINAL VALID DATA  : {len(df)} rows")
    return df


## 4. Eksekusi Penggabungan Keseluruhan (Full Merge & Save)

In [ ]:
# 1. Eksekusi Program Makan Bergizi
df_makan = process_pipeline(RAW_DATA_MAKAN_DIR, "makan_bergizi_gratis", KEYWORDS_MAKAN)

# 2. Eksekusi MBG
df_mbg = process_pipeline(RAW_DATA_MBG_DIR, "mbg", KEYWORDS_MBG)

# 3. Penggabungan Menjadi Merged Full Dataset
df_full = pd.concat([df_makan, df_mbg], axis=0, ignore_index=True)

# Simpan Data Tiga File
output_makan_path = os.path.join(MERGED_OUTPUT_DIR, "merged_makan_bergizi_gratis.csv")
output_mbg_path = os.path.join(MERGED_OUTPUT_DIR, "merged_mbg.csv")
output_full_path = os.path.join(MERGED_OUTPUT_DIR, "merged_mbg_full.csv")

if not df_makan.empty:
    df_makan.to_csv(output_makan_path, index=False)
    print(f"\n💾 Saved Part 1: {output_makan_path}")

if not df_mbg.empty:
    df_mbg.to_csv(output_mbg_path, index=False)
    print(f"💾 Saved Part 2: {output_mbg_path}")

if not df_full.empty:
    df_full.to_csv(output_full_path, index=False)
    print(f"💾 Saved Merged Full: {output_full_path}")


## 5. Preview & Informasi Dataset Lengkap

In [ ]:
print("\n===== INFORMASI DATASET =====")

# Kolom
print("\nKolom dalam dataset:")
print(df_full.columns.tolist())

# Info umum
print("\nInfo dataset:")
df_full.info()

# Total data
print("\nTotal jumlah data:", len(df_full))

# Missing values
print("\nJumlah missing values per kolom:")
print(df_full.isnull().sum())

# Distribusi source
print("\nDistribusi source_keyword:")
print(df_full["source_keyword"].value_counts())

# Cek duplikat total (karena penggabungan mungkin redundan)
duplicate_count = df_full.duplicated(subset=['full_text']).sum()
print(f"\nJumlah data duplikat antar dataset (mbg vs makan): {duplicate_count}")
if duplicate_count > 0:
    df_full.drop_duplicates(subset=['full_text'], keep='first', inplace=True)
    print(f"✨ Duplikat dihapus. Total Data Terakhir: {len(df_full)}")

print("\n===== 5 DATA AWAL =====")
display(df_full.head(5))

print("\n===== 5 DATA AKHIR =====")
display(df_full.tail(5))


## 6. Visualisasi Data

In [ ]:
def plot_timeline(df_list, labels):
    plt.figure(figsize=(15, 6))

    for df, label in zip(df_list, labels):
        if df.empty: continue
        daily_counts = df.set_index('created_at').resample('D').size()
        plt.plot(daily_counts.index, daily_counts.values, label=f"{label}", alpha=0.8)

    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=14))

    plt.title("Daily Tweet Volume Trend")
    plt.xlabel("Date")
    plt.ylabel("Number of Tweets")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def plot_monthly(df_list, labels):
    combined_monthly = pd.DataFrame()

    for df, label in zip(df_list, labels):
        if df.empty: continue
        monthly_counts = df.set_index('created_at').resample('M', kind='period').size()

        # Konversi index period menjadi stamp awal bulan hanya untuk charting string
        monthly_counts.index = monthly_counts.index.strftime('%b %y')
        combined_monthly[label] = monthly_counts

    if combined_monthly.empty:
        print("⚠️ No data available for monthly chart.")
        return

    ax = combined_monthly.plot(kind='bar', figsize=(12, 6), width=0.8, alpha=0.85)

    plt.title("Monthly Tweet Volume Trend")
    plt.xlabel("Month")
    plt.ylabel("Number of Tweets")
    plt.xticks(rotation=0)
    plt.grid(axis='y', alpha=0.3)
    plt.legend()

    for p in ax.patches:
        height = p.get_height()
        if pd.notnull(height) and height > 0:
            ax.annotate(str(int(height)), (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom', xytext=(0, 5), textcoords='offset points', fontsize=9)

    plt.tight_layout()
    plt.show()


In [ ]:
# Detail Full Dataset
print("\n📊 Menampilkan Tren Keseluruhan Dataset Tergabung (Merged Full):\n")
plot_timeline([df_full], ['Merged Full Dataset'])
plot_monthly([df_full], ['Merged Full Dataset'])

In [ ]:
# Detail Dataset Keyword 1 dan 2
print("\n📊 Menampilkan Tren Keseluruhan Dataset Tergabung (Merged Full):\n")
plot_timeline([df_makan, df_mbg], ['Makan Bergizi', 'MBG'])
plot_monthly([df_makan, df_mbg], ['Makan Bergizi', 'MBG'])